# Sous le capot — ce qui se passe vraiment en Python

Ce notebook répond à la question que tout dev devrait pouvoir expliquer :  
**"Pourquoi Python fait-il ça comme ça ?"**

On ne va pas dans le code source de Python — on explique les mécanismes importants  
avec des analogies simples et des démonstrations concrètes.

### Ce qu'on va voir
1. Python vs C — pourquoi Python est "lent" (et pourquoi ça ne pose généralement pas de problème)
2. Le hachage — pourquoi `set` et `dict` sont en O(1)
3. La mémoire — comment Python sait quand libérer un objet
4. Les objets C derrière les types Python — `list`, `dict`, `str`
5. Le GIL — pourquoi Python ne parallélise pas vraiment

---
## 1. Python vs C — le langage interprété

### La différence fondamentale

Un programme C est **compilé** : le code source est transformé en instructions machine  
(des 0 et des 1 que le processeur exécute directement) **avant** de tourner.

Python est **interprété** : le code source est lu et exécuté **ligne par ligne**  
par un programme appelé **l'interpréteur** (CPython, écrit en C).

```
Code Python → interpréteur CPython → instructions machine
Code C      →         compilateur  → instructions machine
```

### Ce que ça coûte

Pour chaque ligne Python, l'interpréteur doit :
1. Lire et décoder l'instruction
2. Déterminer le type de chaque variable (Python ne le sait pas à l'avance)
3. Appeler la bonne fonction C selon le type
4. Gérer le comptage de références (voir section mémoire)

Un entier Python n'est **pas** un entier C.  
`x = 42` en Python crée un **objet Python** qui contient :
- Un compteur de références
- Un pointeur vers le type (`int`)
- La valeur (`42`)

Soit environ **28 bytes** pour un simple entier, contre **4 bytes** en C.

In [ ]:
import sys

# Un entier Python est un objet complet — bien plus gros qu'un int C
print(f"Taille de l'entier 0  : {sys.getsizeof(0)} bytes")
print(f"Taille de l'entier 42 : {sys.getsizeof(42)} bytes")
print(f"Taille de l'entier 1_000_000 : {sys.getsizeof(1_000_000)} bytes")
print()

# En C, un int fait 4 bytes. En Python, le moindre entier fait ~28 bytes
# Parce qu'il porte avec lui : type + compteur de refs + valeur

# Taille d'une liste vide vs liste avec des éléments
liste_vide   = []
liste_100    = list(range(100))
liste_10000  = list(range(10_000))

print(f"Liste vide      : {sys.getsizeof(liste_vide)} bytes")
print(f"Liste de 100    : {sys.getsizeof(liste_100)} bytes")
print(f"Liste de 10 000 : {sys.getsizeof(liste_10000)} bytes")
print()
print("Note : getsizeof retourne la taille du conteneur (les pointeurs),")
print("pas la taille totale du contenu.

### Pourquoi Python reste populaire malgré ça ?

1. **La plupart des programmes ne sont pas CPU-bound** : ils attendent (réseau, disque, base de données). Python n'est pas plus lent que Java pour attendre.

2. **Les bibliothèques critiques sont en C** : NumPy, Pandas, les parsers JSON, les clients HTTP — tout ce qui compte est déjà du C sous le capot.

3. **La productivité prime** : écrire en Python prend 3-5x moins de temps qu'en C. Si le code est assez rapide, ça suffit.

4. **On peut appeler du C depuis Python** : si une fonction Python est un goulot d'étranglement, on la réécrit en C ou en Cython — le reste reste en Python.

> **En entretien** : si on te demande "Python est-il lent ?", la bonne réponse c'est :  
> "Python en lui-même est plus lent que C, mais dans la pratique les libs critiques sont en C, et pour la majorité des applications la performance de Python est suffisante." 

---
## 2. Le hachage — pourquoi `set` et `dict` sont O(1)

### L'analogie des casiers

Imagine 1000 casiers numérotés de 0 à 999.  
Tu veux ranger un objet et le retrouver instantanément.

Au lieu de mettre l'objet dans le premier casier disponible et chercher partout,  
tu **calcules le numéro du casier** à partir de l'objet lui-même : c'est le **hash**.

```python
hash("alice")  →  un nombre entier (ex: 4321567)
4321567 % 1000 →  567   ← numéro du casier
```

Pour retrouver "alice" : recalcule le hash, va directement au casier 567. **O(1)**.  
Peu importe s'il y a 10 ou 10 millions d'autres objets — toujours un seul calcul.

### Qu'est-ce qui est hashable ?

Un objet est **hashable** si :
1. On peut calculer son hash (`hash(obj)` ne plante pas)
2. Son hash **ne change jamais** — un objet mutable ne peut pas être hashable

| Hashable ✓ | Non hashable ✗ |
|---|---|
| `int`, `float`, `str` | `list` |
| `tuple` (si son contenu est hashable) | `dict` |
| `frozenset` | `set` |

C'est pour ça qu'on ne peut pas mettre une liste dans un set ou comme clé de dict : son hash changerait si on la modifiait, et Python ne retrouverait plus son casier.

In [ ]:
# La fonction hash() — le cœur du mécanisme
print("hash de quelques objets :")
print(f"  hash(42)        = {hash(42)}")
print(f"  hash('alice')   = {hash('alice')}")
print(f"  hash((1, 2, 3)) = {hash((1, 2, 3))}")   # tuple → hashable
print()

# Même valeur = même hash (toujours)
print(f"  hash(1) == hash(1.0) : {hash(1) == hash(1.0)}")  # True ! (1 == 1.0 en Python)
print()

# Un objet mutable n'est pas hashable
try:
    hash([1, 2, 3])    # liste → mutable → pas hashable
except TypeError as e:
    print(f"Erreur liste : {e}")

try:
    hash({"a": 1})     # dict → mutable → pas hashable
except TypeError as e:
    print(f"Erreur dict  : {e}")

print()
# frozenset = version immutable de set → hashable → peut être clé de dict
fs = frozenset([1, 2, 3])
print(f"  hash(frozenset) = {hash(fs)}")   # fonctionne

In [ ]:
import timeit

# Démonstration concrète O(1) vs O(n)
# Peu importe la taille, le set trouve en temps constant

for n in [100, 10_000, 1_000_000]:
    data_list = list(range(n))
    data_set  = set(data_list)
    cible = n - 1   # pire cas pour la liste : tout à la fin

    t_list = timeit.timeit(lambda: cible in data_list, number=500)
    t_set  = timeit.timeit(lambda: cible in data_set,  number=500)

    print(f"n={n:>8} → list: {t_list:.5f}s | set: {t_set:.6f}s | ratio: {t_list/t_set:.0f}x")

print()
print("→ Le temps du set reste stable quelle que soit la taille (O(1))")
print("→ Le temps de la liste grandit avec n (O(n))")

### Collisions

Parfois deux objets différents ont le même numéro de casier — c'est une **collision**.  
Python gère ça en chaînant les éléments dans le même casier.  
Avec beaucoup de collisions, les performances se dégradent vers O(n).  
En pratique, Python redimensionne automatiquement la table pour garder les collisions rares.

---
## 3. La gestion mémoire — reference counting

### Comment Python sait quand libérer un objet ?

Chaque objet Python porte un **compteur de références** : le nombre de variables (ou containers) qui pointent vers lui.

```python
x = [1, 2, 3]   # compteur = 1 (x pointe vers la liste)
y = x            # compteur = 2 (x et y pointent vers la même liste)
del x            # compteur = 1 (seulement y)
del y            # compteur = 0 → Python libère la mémoire immédiatement
```

Quand le compteur tombe à **0**, Python libère la mémoire **immédiatement**.  
Pas besoin d'attendre un garbage collector qui tourne de temps en temps (comme en Java).

C'est rapide et prévisible — mais il y a un cas que ça ne gère pas.

In [ ]:
import sys

# Voir le compteur de références en temps réel
x = [1, 2, 3]
print(f"Compteur après création  : {sys.getrefcount(x)}")
# Note : getrefcount ajoute lui-même 1 (l'argument de la fonction est une ref)

y = x
print(f"Compteur après y = x     : {sys.getrefcount(x)}")

liste_de_listes = [x, x, x]
print(f"Compteur dans une liste  : {sys.getrefcount(x)}")

del y
print(f"Compteur après del y     : {sys.getrefcount(x)}")

del liste_de_listes
print(f"Compteur après del liste : {sys.getrefcount(x)}")

### Les références circulaires — le seul cas problématique

```python
a = {}
b = {}
a["ref"] = b    # a pointe vers b
b["ref"] = a    # b pointe vers a
del a
del b
# Les deux objets ont encore un compteur > 0 (ils se pointent mutuellement)
# → fuite mémoire si Python n'avait que le reference counting
```

C'est pour ça que Python a **aussi** un garbage collector cyclique (module `gc`)  
qui détecte ces cycles et les nettoie. Il tourne en arrière-plan, beaucoup moins souvent.

> **En entretien** : "Python gère la mémoire avec le **reference counting** (libération immédiate à 0)  
> complété par un **garbage collector cyclique** pour les références circulaires." 

In [ ]:
import gc

# Compter les objets dans chaque génération du GC
# Python classe les objets en 3 générations selon leur âge
# Plus un objet survit longtemps, plus il est inspecté rarement (optimisation)

print("Statistiques du garbage collector :")
print(f"  Générations : {gc.get_count()}")
# (génération 0, génération 1, génération 2)
# génération 0 = nouveaux objets — inspectés souvent
# génération 2 = vieux objets — inspectés rarement

# Forcer un cycle complet
gc.collect()
print(f"  Après collect : {gc.get_count()}")
print()

# Démonstration d'interning — Python optimise les petits entiers
# Les entiers de -5 à 256 sont pré-créés une seule fois et partagés
a = 100
b = 100
print(f"a is b pour 100  : {a is b}")   # True — même objet en mémoire

a = 1000
b = 1000
print(f"a is b pour 1000 : {a is b}")   # False — deux objets distincts

---
## 4. Les types Python sont des structures C

### `list` — tableau dynamique de pointeurs

Une liste Python est un **tableau C de pointeurs** vers des objets Python.  
Ce n'est pas un tableau d'entiers comme en C — c'est un tableau d'adresses mémoire.

```
liste = [1, "a", True]
        ↓    ↓    ↓
      obj  obj  obj   ← chaque case contient une adresse vers un objet Python
```

Conséquences :
- Accès par index en **O(1)** : aller à la Nième adresse est instantané
- `insert(0, x)` en **O(n)** : décaler toutes les adresses d'un cran
- Une liste peut mélanger n'importe quels types (chaque case est juste un pointeur)

La liste **se redimensionne automatiquement** quand elle est pleine.  
Python ne crée pas une nouvelle liste de taille+1 à chaque `append` — il sur-alloue  
(prévoie de la place pour les prochains `append`), puis agrandit d'un coup.

In [ ]:
import sys

# Observer l'over-allocation : la capacité réelle vs le nombre d'éléments
liste = []
ancienne_taille = sys.getsizeof(liste)

print(f"{'Éléments':<10} {'Taille (bytes)':<18} {'Saut ?'}")
print("-" * 40)

for i in range(17):
    liste.append(i)
    nouvelle_taille = sys.getsizeof(liste)
    saut = "← réallocation" if nouvelle_taille != ancienne_taille else ""
    print(f"{i+1:<10} {nouvelle_taille:<18} {saut}")
    ancienne_taille = nouvelle_taille

print()
print("Python sur-alloue : il ne réalloue pas à chaque append")
print("→ c'est pour ça que append est O(1) amorti

### `dict` — table de hachage en C

Un dict Python est une **table de hachage** : un grand tableau C d'entrées  
`(hash, clé, valeur)`.

À partir de Python 3.7, les dicts sont aussi **ordonnés** (ordre d'insertion garanti)  
grâce à une optimisation : un tableau compact des indices + le tableau des entrées.

Redimensionnement automatique quand le taux de remplissage dépasse 2/3  
(pour éviter trop de collisions).

In [ ]:
import sys

# Taille d'un dict selon le nombre d'entrées
d = {}
print(f"dict vide        : {sys.getsizeof(d)} bytes")

d = {"a": 1}
print(f"dict 1 entrée    : {sys.getsizeof(d)} bytes")

d = {i: i for i in range(8)}
print(f"dict 8 entrées   : {sys.getsizeof(d)} bytes")

d = {i: i for i in range(9)}
print(f"dict 9 entrées   : {sys.getsizeof(d)} bytes")   # réallocation ici

d = {i: i for i in range(100)}
print(f"dict 100 entrées : {sys.getsizeof(d)} bytes")

print()
print("Les sauts correspondent aux réallocations (le taux de remplissage dépasse 2/3)")

### `str` — séquence immuable de caractères Unicode

Les strings Python 3 sont en **Unicode** (UTF-8 / Latin-1 / UCS-2 / UCS-4 selon le contenu).  
Python choisit automatiquement l'encodage interne le plus compact selon les caractères présents.

**Immuable** = une fois créée, une string ne peut pas être modifiée.  
`s += "x"` crée une **nouvelle string** — pas une modification en place.

Conséquence pratique : accumuler des strings avec `+=` dans une boucle est O(n²).  
Solution : `"".join(liste_de_strings)` qui alloue une seule fois.

In [ ]:
import sys

# Taille des strings selon le contenu
s_ascii   = "hello"           # Latin-1 : 1 byte par caractère
s_latin   = "héllo"           # Latin-1 : 1 byte par caractère (é inclus)
s_unicode = "héllo 🐍"        # UCS-4 : 4 bytes par caractère (à cause de l'emoji)

print(f"'hello' (5 cars ASCII)   : {sys.getsizeof(s_ascii)} bytes → {(sys.getsizeof(s_ascii)-49)/5:.1f} byte/car")
print(f"'héllo' (5 cars Latin-1) : {sys.getsizeof(s_latin)} bytes → {(sys.getsizeof(s_latin)-49)/5:.1f} byte/car")
print(f"'héllo 🐍' (7 cars UCS4) : {sys.getsizeof(s_unicode)} bytes → {(sys.getsizeof(s_unicode)-76)/7:.0f} bytes/car")
print()

# Interning des strings — Python partage les strings "simples"
a = "bonjour"
b = "bonjour"
print(f"'bonjour' is 'bonjour' : {a is b}")  # True — même objet en mémoire

a = "bonjour monde"
b = "bonjour monde"
print(f"'bonjour monde' is ... : {a is b}")  # Peut varier selon l'implémentation

---
## 5. Le GIL — Global Interpreter Lock

### En une phrase

Le **GIL** est un verrou qui empêche deux threads Python de s'exécuter **simultanément**.  
Même sur un processeur 8 cœurs, un programme Python ne fait tourner qu'**un seul thread à la fois**.

### Pourquoi il existe

Le reference counting (compteur de références) n'est **pas thread-safe** sans protection.  
Si deux threads modifient le compteur d'un même objet en même temps → corruption mémoire.  
Le GIL est la solution simple : un seul thread accède à l'interpréteur à la fois.

### Les conséquences pratiques

| Cas | Impact du GIL |
|---|---|
| Code **I/O-bound** (réseau, disque, BD) | **Aucun** — le thread libère le GIL pendant les attentes |
| Code **CPU-bound** (calcul, algo) | **Bloquant** — les threads ne parallélisent pas vraiment |

### La solution pour le CPU-bound

- **`multiprocessing`** : plusieurs processus Python (chacun a son GIL) → vrai parallélisme
- **NumPy, Cython** : s'exécutent en C, hors du GIL
- **Python 3.13+** : le GIL devient optionnel (expérimental)

> **En entretien** : "Le GIL empêche le vrai parallélisme entre threads en Python.  
> Pour le code CPU-intensif, on utilise `multiprocessing` ou des librairies C comme NumPy.  
> Pour le code I/O (web, réseau), `asyncio` ou `threading` suffisent car le GIL est libéré pendant les attentes." 

In [ ]:
import threading
import time
import timeit

# Démonstration : les threads Python ne parallélisent pas le CPU

def calcul_intensif(n):
    """Calcul CPU-bound : somme des carrés"""
    return sum(i**2 for i in range(n))

N = 5_000_000

# Version séquentielle
debut = time.perf_counter()
calcul_intensif(N)
calcul_intensif(N)
seq = time.perf_counter() - debut

# Version avec 2 threads (même CPU à cause du GIL)
debut = time.perf_counter()
t1 = threading.Thread(target=calcul_intensif, args=(N,))
t2 = threading.Thread(target=calcul_intensif, args=(N,))
t1.start(); t2.start()
t1.join();  t2.join()
thread = time.perf_counter() - debut

print(f"Séquentiel  : {seq:.2f}s")
print(f"2 threads   : {thread:.2f}s")
print(f"Ratio       : {thread/seq:.2f}x")
print()
print("→ Les threads ne sont pas plus rapides (GIL) — parfois même plus lents")
print("  (overhead de création et de synchronisation des threads)")
print()
print("Pour vraiement paralléliser : utiliser multiprocessing")

---
## Récapitulatif — ce qu'il faut retenir

### Python vs C
Python est interprété → chaque ligne passe par l'interpréteur CPython.  
Un entier Python ≈ 28 bytes (objet complet) vs 4 bytes en C.  
En pratique : les libs critiques (NumPy, etc.) sont en C sous le capot.

### Hachage
`set` et `dict` calculent un hash → accès direct au "casier" → **O(1)**.  
Seuls les objets **immuables** sont hashables (pas les listes ni les dicts).  
`tuple`, `str`, `int`, `frozenset` → hashables.

### Mémoire
**Reference counting** : chaque objet compte ses références.  
À 0 → libération immédiate.  
Cycles → gérés par le garbage collector (`gc`).  
Entiers -5 à 256 → pré-créés et partagés (interning).

### Types internes
- `list` → tableau C de pointeurs (over-alloue pour les appends)
- `dict` → table de hachage (redimensionnée à 2/3 de remplissage)
- `str` → Unicode immuable (Latin-1 / UCS-2 / UCS-4 selon le contenu)

### GIL
Un seul thread Python à la fois.  
I/O-bound → `threading` ou `asyncio` (le GIL est libéré pendant les attentes).  
CPU-bound → `multiprocessing` ou libs C (NumPy) qui s'exécutent hors du GIL.